In [1]:
import numpy as np
import matplotlib.pyplot as plt
import arviz as az
import pickle

from seir_model.simulator import SEIRSimulator

# --- Load posterior samples ---
with open("results/posterior_samples.pkl", "rb") as f:
    trace = pickle.load(f)

# --- Simulation settings ---
population = 1_000_000
n_draws = 100
n_days = 100

# --- Hypothetical scenario: 50% reduction in beta (e.g. due to strong intervention) ---
scenario_results = []

for i in range(n_draws):
    sample = trace[i]
    params = {
        "beta": sample["beta"] * 0.5,  # hypothetical intervention
        "incubation": sample["incubation"],
        "symptomatic": sample["symptomatic"],
        "h_rate": sample["h_rate"],
        "h_duration": sample["h_duration"],
        "icu_rate": sample["icu_rate"],
        "icu_duration": sample["icu_duration"],
        "mu": sample["mu"]
    }
    I0 = sample["I0"]
    E0 = sample["E0"]
    S0 = population - E0 - I0
    initial_state = [S0, E0, I0, 0, 0, 0, 0]

    sim = SEIRSimulator(N=population, params=params, stochastic=False)
    res = sim.simulate(initial_state, n_days)
    scenario_results.append(res)

scenario_results = np.stack(scenario_results)  # shape (n_draws, n_days+1, 7)

# --- Compute statistics ---
ICU_traces = scenario_results[:, :, 4]  # ICU compartment
mean_ICU = ICU_traces.mean(axis=0)
low_ICU, high_ICU = np.percentile(ICU_traces, [2.5, 97.5], axis=0)

# --- Plot results ---
days = np.arange(n_days + 1)

plt.figure(figsize=(10, 5))
plt.fill_between(days, low_ICU, high_ICU, alpha=0.3, label="95% CI")
plt.plot(days, mean_ICU, label="Predicted ICU (mean)", color="black")
plt.xlabel("Day")
plt.ylabel("ICU patients")
plt.title("Predictive scenario: 50% reduction in transmission")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'seir_model'